In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from homeomorphism.hooks import list_module_names, get_submodule, get_submodules, get_modules, register_forward_capture_hooks, submodule_output_jacobian

In [2]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
cache = {}
modules = get_modules(model, ["attn"])
handles = register_forward_capture_hooks(modules, cache, "attn")

_ = model(**tokenizer("Hello what's up", return_tensors="pt"))

print(cache.keys())           # dict_keys(["attn_0", "attn_1", ...])
print(cache["attn_0"].shape)  # tensor shape for first hooked module

# remove_hooks(handles)

dict_keys(['attn_0', 'attn_1', 'attn_2', 'attn_3', 'attn_4', 'attn_5', 'attn_6', 'attn_7', 'attn_8', 'attn_9', 'attn_10', 'attn_11'])
torch.Size([1, 4, 768])


In [9]:
inputs = tokenizer("H ", return_tensors="pt")
jac = submodule_output_jacobian(
    model,
    "transformer.h.0.attn",
    inputs,
    input_index=0,
    batch_index=0,
    vectorize=True,
 )

print(jac.shape)

torch.Size([1, 2, 768, 1, 2, 768])
